In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

KeyboardInterrupt: 

In [ ]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [ ]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download  
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and start a local model:
   ```bash
   ollama run llama3
   ```

   This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. To test that the local server is running:
   ```bash
   curl http://localhost:11434
   ```

   You should get a response like:
   ```json
   {"models": [...]}  
   ```

If you want to use it from Python, install the client with:

```bash
pip install ollama
```

Then you can call it with:

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": your_prompt}]
)

print(response['message']['content'])
```


In [ ]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see any FAQ entry about **Olama/Ollama** specifically.

The closest relevant guidance is that you **can run the course locally** instead of using Codespaces, as long as you’re comfortable setting up the needed tools like:

- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

If you run locally, make sure to **document your setup** and keep the environment **reproducible**.

If you meant **Ollama** specifically, the context here doesn’t include instructions for it.


In [4]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'You can absolutely try to join it — whether you’re able to depends on the course’s enrollment rules and deadlines.\n\nHere’s the quickest way to check:\n1. **Look for the course page or syllabus** for enrollment requirements.\n2. **Check the registration deadline** or whether the course is still open.\n3. **Contact the instructor or department** and ask if late enrollment is allowed.\n4. If it’s full, ask whether there’s a **waitlist** or permission code process.\n\nIf you want, I can help you draft a short message to the instructor asking to join.'

In [5]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [8]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [9]:
len(response.output)

1

In [12]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it started? discovered the course can I join it late enrollment"}', call_id='call_tVKdk8QKBB7mnDLmMTo78sxk', name='search', type='function_call', id='fc_0d87336b00bb7331006a3608b78180819db0b29be59deafe18', namespace=None, status='completed')]

In [13]:
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it started? discovered the course can I join it late enrollment"}', call_id='call_tVKdk8QKBB7mnDLmMTo78sxk', name='search', type='function_call', id='fc_0d87336b00bb7331006a3608b78180819db0b29be59deafe18', namespace=None, status='completed')